# Belief in physics: data → transformer → analysis

One self-contained experimental notebook in three phases. It trains from the current physics implementation only; old checkpoints and old result files are never loaded.

### Setup and experiment switches

This cell imports the existing physics, transformer, and analysis helpers, then defines every top-level experiment choice. Change `SYSTEM_NAME`, `INITIAL_STATE_MODE`, or `TRAINING_RUNS` here; smoke mode reduces the workload without changing the pipeline.

In [ ]:
%matplotlib inline

import json, os
from dataclasses import asdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Dataset

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
import sys
sys.path.insert(0, str(ROOT))

from models.analysis import residual_streams_batched, tick_features
from models.explore import next_token_probs
from models.train import TrainConfig, pick_device, train
from models.transformer import ModelConfig, TinyTransformer
from physics import visualise as V
from physics.messk import simplex_embedding
from physics.messk_configs import MESSK_CONFIGS, make_process

# ---------------- top-level experiment switches ----------------
SYSTEM_NAME = 'pendulum'  # pendulum | predator_prey | sphere | double_pendulum
HMM_TYPE = 'mess4'        # deliberately fixed to Mess-4 for this notebook
INITIAL_STATE_MODE = 'fixed'  # fixed | random
INITIAL_JITTER_STD = 0.10
DATA_SEED = 20260916
M_TICKS = 16
STEPS_PER_TICK = 10

SMOKE = os.environ.get('BELIEF_NOTEBOOK_SMOKE') == '1'
DEVICE = os.environ.get('BELIEF_NOTEBOOK_DEVICE', 'cpu' if SMOKE else pick_device())
N_DATA_SEQUENCES = 32 if SMOKE else 256
N_ANALYSIS_SEQUENCES = 64 if SMOKE else 1024
CV_FOLDS = 2 if SMOKE else 5
RIDGE_ALPHAS = (1.0, 10.0) if SMOKE else (0.1, 1.0, 10.0, 100.0, 1000.0)
OUTPUT_DIR = Path(os.environ.get('BELIEF_NOTEBOOK_OUTPUT_DIR', ROOT / 'experiments' / 'outputs-three-phase'))

# Add dictionaries here to sweep seeds or architectures. One configuration for now.
TRAINING_RUNS = [{
    'name': 'single_config',
    'seed': 0,
    'd_model': 32 if SMOKE else 64,
    'n_layers': 1 if SMOKE else 2,
    'n_heads': 1,
    'd_mlp': 64 if SMOKE else 256,
    'batch_size': 16 if SMOKE else 128,
    'total_tokens': 80_000 if SMOKE else 2_000_000,
    'learning_rate': 1e-3,
}]

assert HMM_TYPE == 'mess4'
PROCESS_NAME = f'{SYSTEM_NAME}_{HMM_TYPE}'
assert PROCESS_NAME in MESSK_CONFIGS, (PROCESS_NAME, sorted(MESSK_CONFIGS))
print(f'{PROCESS_NAME=} {INITIAL_STATE_MODE=} {DEVICE=} {SMOKE=}')

## Phase 1 — data

This phase builds the selected physics–Mess-4 process and verifies the complete path from hidden state to model input. The HMM emits one of four letters; each letter applies a distinct perturbation, the physical system evolves for `STEPS_PER_TICK`, and the observable is discretized into transformer tokens.

The plots expose both the physical trajectory and the hidden process, while the checks cover bin usage, clipping, finite dynamics, belief normalization, stability, and tensor shapes. The exact predictive belief is retained only as a later analysis target—the transformer never receives it.

### Generate a physics batch

This cell creates the selected process and samples trajectories. Fixed mode uses the system default state; random mode adds bounded Gaussian jitter before calling the shared `sample_batch` implementation.

In [ ]:
proc = make_process(PROCESS_NAME, m=M_TICKS, n_steps=STEPS_PER_TICK)

def make_initial_states(proc, rng, n):
    z = proc.system.initial_state(n).astype(np.float64, copy=True)
    if INITIAL_STATE_MODE == 'fixed':
        return z
    if INITIAL_STATE_MODE != 'random':
        raise ValueError(f'unknown INITIAL_STATE_MODE {INITIAL_STATE_MODE!r}')
    z += rng.normal(0.0, INITIAL_JITTER_STD, size=z.shape)
    names = tuple(proc.system.state_names)
    if hasattr(proc.system, 'log_bound'):
        z = np.clip(z, -proc.system.log_bound, proc.system.log_bound)
    if hasattr(proc.system, 'omega_max'):
        for i, name in enumerate(names):
            if 'omega' in name:
                z[:, i] = np.clip(z[:, i], -proc.system.omega_max, proc.system.omega_max)
    if hasattr(proc.system, 'rate_max'):
        from physics.systems.sphere import THETA_MAX, THETA_MIN
        z[:, names.index('theta')] = np.clip(z[:, names.index('theta')], THETA_MIN, THETA_MAX)
        z[:, names.index('psi')] = (z[:, names.index('psi')] + np.pi) % (2 * np.pi) - np.pi
        for name in ('dtheta', 'dpsi'):
            z[:, names.index(name)] = np.clip(z[:, names.index(name)], -proc.system.rate_max, proc.system.rate_max)
    return z

def sample_physics_batch(proc, rng, n):
    z0 = make_initial_states(proc, rng, n)
    batch = proc.sample_batch(rng, n, initial_state=z0)
    return {**batch, 'initial_state': z0}

batch = sample_physics_batch(proc, np.random.default_rng(DATA_SEED), N_DATA_SEQUENCES)
print({k: v.shape for k, v in batch.items()})
print('fixed baseline:', dict(zip(proc.system.state_names, proc.system.initial_state(1)[0])))

### Visualize a physical trajectory

This plot aligns observables, physics metrics, and discretized tokens for one sequence. Colored vertical lines mark the perturbation chosen by Mess-4 at each cycle.

In [ ]:
# Physics trajectory example generated from the selected initial-state mode.
i = 0
steps = np.arange(proc.seq_len)
fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True)
for c, name in enumerate(proc.channel_names):
    axes[0].plot(steps, batch['observable'][i, :, c], label=name)
for c, name in enumerate(proc.system.metric_names):
    axes[1].plot(steps, batch['metric'][i, :, c], label=name)
axes[2].step(steps, batch['tokens'][i], where='mid', label='token')
for ax in axes:
    for tick, letter in enumerate(batch['letters'][i]):
        ax.axvline(tick * proc.n_steps, color=f'C{letter}', alpha=0.16)
    ax.grid(alpha=0.25); ax.legend(loc='upper right')
axes[0].set_title(f'{PROCESS_NAME}: trajectory; line colors mark Mess-4 perturbations')
axes[-1].set_xlabel('physics step')
plt.tight_layout()
plt.show()

### Visualize the HMM and exact belief

This cell draws the Mess-4 state graph, transition and emission matrices, and the exact predictive belief over hidden moods. It also prints the sampled hidden moods and emitted perturbation letters for direct inspection.

In [ ]:
# HMM transition graph, matrices, sampled outputs, and exact belief trajectory.
chain = proc.chain
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
angles = np.linspace(0, 2 * np.pi, chain.n_states, endpoint=False)
xy = np.c_[np.cos(angles), np.sin(angles)]
ax = axes[0]
for state, (x, y) in enumerate(xy):
    ax.scatter(x, y, s=900, color=f'C{state}', zorder=3)
    ax.text(x, y, str(state), ha='center', va='center', color='white', weight='bold')
for source in range(chain.n_states):
    for target in range(chain.n_states):
        if source == target:
            continue
        ax.annotate('', xy=xy[target], xytext=xy[source],
                    arrowprops={'arrowstyle': '->', 'alpha': 0.22, 'connectionstyle': 'arc3,rad=.12'})
ax.set_title(f'Mess-4 graph\nstay={chain.stay:.2f}, other={chain.x:.2f}')
ax.set_axis_off(); ax.set_aspect('equal')
for matrix, title, matrix_ax in ((chain.T, 'transition T', axes[1]), (chain.E, 'emission E', axes[2])):
    im = matrix_ax.imshow(matrix, vmin=0, vmax=1, cmap='Blues')
    for row in range(chain.n_states):
        for col in range(chain.n_states):
            matrix_ax.text(col, row, f'{matrix[row, col]:.2f}', ha='center', va='center')
    matrix_ax.set_title(title); matrix_ax.set_xlabel('to / letter'); matrix_ax.set_ylabel('from / mood')
belief_tick = batch['beliefs'][0, proc.n_steps - 1::proc.n_steps]
for state in range(chain.n_states):
    axes[3].step(np.arange(proc.m), belief_tick[:, state], where='post', label=f'mood {state}')
axes[3].set_ylim(0, 1); axes[3].set_title('exact predictive belief'); axes[3].legend(fontsize=8)
plt.tight_layout(); plt.show()
print('hidden moods:', batch['states'][0].tolist())
print('emitted letters:', batch['letters'][0].tolist())

### Check coverage, beliefs, and stability

This cell summarizes vocabulary coverage, clipping, belief entropy and normalization, memory length, and simulator stability. Assertions stop the notebook if probabilities, token ranges, or physical values are invalid.

In [ ]:
# Physics/HMM statistics and stability checks.
bins = proc.bin_report(batch['observable'])
baseline_trace = V.trace(proc, seed=DATA_SEED)
stable = V.stability(baseline_trace)
belief = batch['beliefs'].reshape(-1, chain.n_states)
stats = {
    'system': PROCESS_NAME, 'hmm': HMM_TYPE, 'initial_state_mode': INITIAL_STATE_MODE,
    'sequences': len(batch['tokens']), 'sequence_length': proc.seq_len,
    'vocabulary': proc.n_obs, 'used_bins': bins['used_bins'], 'clipped_fraction': bins['clipped'],
    'belief_normalization_error': float(np.abs(belief.sum(1) - 1).max()),
    'belief_entropy_fraction': float(-(belief * np.log(np.clip(belief, 1e-12, None))).sum(1).mean() / np.log(chain.n_states)),
    'belief_memory_letters': chain.memory_length(), 'lyapunov': stable['lyapunov'],
    'baseline_stable': stable['stable'], 'stability_reasons': stable['reasons'],
    'dt': proc.dt, 'steps_per_tick': proc.n_steps, 'ticks': proc.m, 'delta_v': proc.delta_v,
}
display(pd.Series(stats, name='value').to_frame())
assert np.allclose(chain.T.sum(1), 1) and np.allclose(chain.E.sum(1), 1)
assert stats['belief_normalization_error'] < 1e-10
assert batch['tokens'].min() >= 0 and batch['tokens'].max() < proc.n_obs
assert np.isfinite(batch['observable']).all() and np.isfinite(batch['metric']).all()

### Package samples for PyTorch

This cell wraps a finite diagnostic batch in a `Dataset` and stacks rows with the correct integer or floating-point dtype. A small `DataLoader` batch verifies the shapes expected downstream.

In [ ]:
# Finite dataset and collator for inspection/evaluation. Training below deliberately streams fresh batches.
class PhysicsDataset(Dataset):
    def __init__(self, proc, n_sequences, seed):
        self.batch = sample_physics_batch(proc, np.random.default_rng(seed), n_sequences)
    def __len__(self):
        return len(self.batch['tokens'])
    def __getitem__(self, index):
        return {name: values[index] for name, values in self.batch.items()}

def physics_collate(rows):
    out = {}
    for name in rows[0]:
        values = np.stack([row[name] for row in rows])
        dtype = torch.long if np.issubdtype(values.dtype, np.integer) else torch.float32
        out[name] = torch.as_tensor(values, dtype=dtype)
    return out

dataset = PhysicsDataset(proc, N_DATA_SEQUENCES, DATA_SEED + 1)
loader = DataLoader(dataset, batch_size=min(8, len(dataset)), shuffle=False, collate_fn=physics_collate)
loader_batch = next(iter(loader))
print({k: (tuple(v.shape), str(v.dtype)) for k, v in loader_batch.items()})
assert loader_batch['tokens'].shape[1:] == (proc.seq_len,)
assert loader_batch['beliefs'].shape[1:] == (proc.seq_len, chain.n_states)
assert loader_batch['tokens'].dtype == torch.long

## Phase 2 — transformer

This phase trains a plain causal transformer to predict the next discretized observation token from earlier tokens. Training streams freshly sampled physics batches, so the model does not repeatedly optimize on the finite diagnostic dataset from Phase 1.

`TRAINING_RUNS` supports multiple seeds or architectures by adding dictionaries, although the default contains one configuration. The phase plots train/evaluation loss and inspects full token probabilities, not just argmax accuracy. Intermediate checkpoints are skipped; each run saves exactly one final transformer.

### Train configurations and save final weights

For each entry in `TRAINING_RUNS`, this cell creates matched trained and random-initialized transformers, trains on fresh token batches, and writes only the final state dictionary plus a JSON metadata sidecar.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
trained_runs = {}
for run in TRAINING_RUNS:
    cfg = ModelConfig(
        vocab_size=proc.n_obs, n_ctx=proc.seq_len, n_layers=run['n_layers'],
        n_heads=run['n_heads'], d_model=run['d_model'], d_mlp=run['d_mlp'], seed=run['seed'],
    )
    random_model = TinyTransformer(cfg)
    model = TinyTransformer(cfg)
    model.load_state_dict(random_model.state_dict())
    report = train(
        model, lambda rng, n: sample_physics_batch(proc, rng, n)['tokens'], proc.seq_len,
        TrainConfig(total_tokens=run['total_tokens'], batch_size=run['batch_size'],
                    learning_rate=run['learning_rate'], seed=run['seed'],
                    log_every=1 if SMOKE else 20, checkpoint_at=()),
        device=DEVICE,
    )
    final_path = OUTPUT_DIR / f"{PROCESS_NAME}_{run['name']}_seed{run['seed']}_final.pt"
    torch.save({k: v.detach().cpu() for k, v in model.state_dict().items()}, final_path)
    metadata = {
        'process': PROCESS_NAME, 'hmm_type': HMM_TYPE, 'initial_state_mode': INITIAL_STATE_MODE,
        'initial_jitter_std': INITIAL_JITTER_STD, 'model': asdict(cfg), 'training': run,
        'tokens_seen': report['tokens_seen'], 'final_loss': report['final'],
    }
    final_path.with_suffix('.json').write_text(json.dumps(metadata, indent=2))
    trained_runs[run['name']] = {'model': model.eval(), 'random_model': random_model.eval(),
                                 'config': cfg, 'report': report, 'path': final_path}
    print(run['name'], 'saved', final_path, report['final'])

assert all(run['path'].exists() for run in trained_runs.values())

### Inspect the loss curve

This cell plots training and evaluation cross-entropy for every run. The dotted `log(vocabulary size)` line is the loss expected from uniform next-token guessing.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for name, run in trained_runs.items():
    h = pd.DataFrame(run['report']['history'])
    ax.plot(h['step'], h['train_loss'], label=f'{name} train')
    ax.plot(h['step'], h['eval_loss'], '--', label=f'{name} eval')
ax.axhline(np.log(proc.n_obs), color='0.4', ls=':', label='uniform')
ax.set(xlabel='optimization step', ylabel='next-token cross entropy', title='training loss')
ax.legend(); ax.grid(alpha=.25); plt.show()

### Inspect actual next-token predictions

This cell evaluates a fresh trajectory, compares true and predicted token IDs and decoded physical values, and plots the complete next-token probability heatmap. Probability normalization and finite negative log-likelihood are checked explicitly.

In [ ]:
# Direct model sanity check: numeric token outputs and a probability heatmap.
chosen = trained_runs[TRAINING_RUNS[0]['name']]
model = chosen['model']
check = sample_physics_batch(proc, np.random.default_rng(DATA_SEED + 2), 4)
tokens = check['tokens'][0]
probs = next_token_probs(model, tokens, DEVICE)
predicted = probs[:-1].argmax(1)
actual = tokens[1:]
nll = -np.log(np.maximum(probs[:-1, :][np.arange(len(actual)), actual], 1e-12))
decoded_actual = proc.undiscretise(actual[:24])
decoded_predicted = proc.undiscretise(predicted[:24])
display(pd.DataFrame({
    'actual_token': actual[:24], 'predicted_token': predicted[:24],
    'actual_value': [np.round(x, 3).tolist() for x in decoded_actual],
    'predicted_value': [np.round(x, 3).tolist() for x in decoded_predicted],
}))
fig, ax = plt.subplots(figsize=(13, 5))
ax.imshow(probs[:-1].T, origin='lower', aspect='auto', cmap='magma')
ax.plot(np.arange(len(actual)), actual, color='cyan', lw=1, label='actual next token')
ax.set(xlabel='position', ylabel='token id', title=f'next-token probability heatmap; mean NLL={nll.mean():.3f}')
ax.legend(); plt.show()
assert probs.shape == (proc.seq_len, proc.n_obs)
assert np.allclose(probs.sum(1), 1.0, atol=1e-5)
assert np.isfinite(nll).all()

## Phase 3 — ridge analysis

This phase asks whether transformer residual streams linearly expose the physical metric or the exact HMM belief. The transformer is frozen: standardized ridge regressions read its activations, with physics and belief fitted and scored separately.

Four feature formats are compared: one layer at one token, all layers at one token, one layer across a perturbation cycle, and all layers across a cycle. A “single token” is the final observation in each cycle. Ridge strength is selected with grouped cross-validation over complete trajectories, then measured on untouched trajectories. Every probe is repeated on the architecture-matched random transformer to establish a baseline.

### Define probe targets, feature views, and splits

This cell extracts end-of-cycle physics and simplex-embedded belief targets, defines the four residual-stream feature formats, and creates disjoint development and test trajectory groups to prevent sequence leakage.

In [ ]:
PROBE_MODES = (
    'single_layer_single_token', 'all_layers_single_token',
    'single_layer_cycle', 'all_layers_cycle',
)
SELECTED_LAYER = -1

def probe_targets(batch, proc):
    last = np.arange(proc.n_steps - 1, proc.seq_len, proc.n_steps)
    belief = batch['beliefs'][:, last] @ simplex_embedding(proc.chain.n_states)
    metric = batch['metric'][:, last]
    return {'physics': metric.reshape(-1, metric.shape[-1]),
            'belief': belief.reshape(-1, belief.shape[-1])}

def probe_features(streams, mode, proc):
    selected = [streams[SELECTED_LAYER]] if mode.startswith('single_layer') else streams
    return tick_features(selected, proc.n_steps, whole_tick=mode.endswith('cycle')).astype(np.float32)

analysis_batch = sample_physics_batch(proc, np.random.default_rng(DATA_SEED + 3), N_ANALYSIS_SEQUENCES)
targets = probe_targets(analysis_batch, proc)
groups = np.repeat(np.arange(N_ANALYSIS_SEQUENCES), proc.m)
order = np.random.default_rng(DATA_SEED + 4).permutation(N_ANALYSIS_SEQUENCES)
test_sequences = set(order[:max(1, N_ANALYSIS_SEQUENCES // 5)].tolist())
test_mask = np.array([g in test_sequences for g in groups])
dev_mask = ~test_mask
assert set(groups[dev_mask]).isdisjoint(set(groups[test_mask]))

### Fit grouped cross-validated ridge probes

This cell standardizes each feature matrix, chooses ridge regularization with `GroupKFold`, and reports untouched test R². It runs every feature format and target on both the trained and random transformers.

In [ ]:
def cross_validated_ridge(X, y, groups, dev_mask, test_mask):
    estimator = make_pipeline(StandardScaler(), Ridge(solver='lsqr'))
    search = GridSearchCV(
        estimator, {'ridge__alpha': RIDGE_ALPHAS},
        cv=GroupKFold(n_splits=CV_FOLDS), scoring='r2', n_jobs=1,
    )
    search.fit(X[dev_mask], y[dev_mask], groups=groups[dev_mask])
    prediction = search.best_estimator_.predict(X[test_mask])
    return {
        'alpha': float(search.best_params_['ridge__alpha']),
        'cv_r2': float(search.best_score_),
        'test_r2': float(r2_score(y[test_mask], prediction, multioutput='uniform_average')),
        'feature_dim': X.shape[1],
        'dev_rows_per_feature': float(dev_mask.sum() / X.shape[1]),
    }

def evaluate_readouts(model, model_name):
    streams = residual_streams_batched(model.to(DEVICE).eval(), analysis_batch['tokens'], DEVICE)
    rows = []
    for mode in PROBE_MODES:
        X = probe_features(streams, mode, proc)
        for target_name in ('physics', 'belief'):
            score = cross_validated_ridge(X, targets[target_name], groups, dev_mask, test_mask)
            rows.append({'model': model_name, 'mode': mode, 'target': target_name, **score})
    return rows

probe_rows = evaluate_readouts(chosen['model'], 'trained')
probe_rows += evaluate_readouts(chosen['random_model'], 'random_init')
probe_results = pd.DataFrame(probe_rows)
display(probe_results.sort_values(['target', 'mode', 'model']).reset_index(drop=True))

### Compare trained and random representations

This cell plots held-out R² by target and feature format, then computes the trained-minus-random improvement. Final assertions confirm that all requested modes, models, and targets produced finite scores.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)
for ax, target in zip(axes, ('physics', 'belief')):
    table = probe_results[probe_results.target == target].pivot(index='mode', columns='model', values='test_r2')
    table.loc[list(PROBE_MODES)].plot.bar(ax=ax)
    ax.axhline(0, color='black', lw=.8); ax.set_title(f'{target} held-out R²'); ax.grid(axis='y', alpha=.25)
plt.tight_layout(); plt.show()

delta = probe_results.pivot(index=['mode', 'target'], columns='model', values='test_r2')
delta['trained_minus_random'] = delta['trained'] - delta['random_init']
display(delta.reset_index())
assert set(probe_results['mode']) == set(PROBE_MODES)
assert set(probe_results['model']) == {'trained', 'random_init'}
assert set(probe_results['target']) == {'physics', 'belief'}
assert np.isfinite(probe_results[['cv_r2', 'test_r2']]).all().all()

## What is saved

Only each run’s final transformer and a small JSON sidecar are written to `experiments/outputs-three-phase/` (or `BELIEF_NOTEBOOK_OUTPUT_DIR`). The random baseline is recreated from the same model seed. Phase 3 is deliberately a sanity analysis; the complete saved-weight analysis belongs in the later analysis notebook.